# Week 1 EDA: the sampled application table

Work through the sections in order. Cells marked **PROVIDED** run as they are. For each
**TODO(student)** section, answer the question with code in the empty cell, then write one
or two sentences of what you see. Keep reusable code in `src/`; this notebook is for looking.

Run it from the repository root with `jupyter notebook` (or `jupyter lab`) after `python run.py sample`.

## 1. Load the sample  (PROVIDED)

In [ ]:
# PROVIDED
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import load_application  # noqa: E402
from src.utils import load_config  # noqa: E402

config = load_config(ROOT / "config" / "config.yaml")
app = load_application(config)
print(app.shape)
app.head()

## 2. Target rate  (PROVIDED)

In [ ]:
# PROVIDED
counts = app["TARGET"].value_counts().sort_index()
print(counts)
print(f"Default rate: {app['TARGET'].mean():.2%}")
counts.plot.bar(title="TARGET (1 = payment difficulties)")
plt.show()

## 3. Missingness  (TODO(student))

Which columns have the most missing values? How many columns are more than half empty?
Do the most-missing columns seem to belong together (look at their names)?

In [ ]:
# Missing values in each application column
missing_pct = app.isna().mean().mul(100).sort_values(ascending=False)
print("Columns with the most missing values (%):")
print(missing_pct.head(15).round(1).to_string())
more_than_half = missing_pct[missing_pct > 50]
print(f"\nColumns more than half empty: {len(more_than_half)}")
print(more_than_half.index.tolist())


## 4. The DAYS_EMPLOYED anomaly  (TODO(student))

Look at the distribution of `DAYS_EMPLOYED`. Which value looks impossible, and how many
applicants have it? Is the default rate of those applicants different from everyone else's?
What does that tell you about how to handle it?

In [ ]:
# Compare the sentinel group with applicants who have a recorded value
sentinel = app["DAYS_EMPLOYED"].eq(365243)
print("Sentinel value: 365243")
print(f"Applicants with sentinel: {int(sentinel.sum())} ({sentinel.mean():.1%})")
print(f"Default rate for sentinel group: {app.loc[sentinel, 'TARGET'].mean():.1%}")
print(f"Default rate for other applicants: {app.loc[~sentinel, 'TARGET'].mean():.1%}")
print("This value is a placeholder, so the model should treat it as missing and keep "
      "a separate indicator for it.")


## 5. Default rate by a categorical column  (TODO(student))

Pick one categorical column (for example `NAME_CONTRACT_TYPE`, `CODE_GENDER`, or
`OCCUPATION_TYPE`). Show the default rate and the number of applicants in each category.
Which differences would you trust, given how few applicants some categories have?

In [ ]:
# Compare counts and target rate by gender, retaining missing values as a category
by_gender = app.groupby("CODE_GENDER", dropna=False)["TARGET"].agg(
    applicants="size", default_rate="mean"
)
by_gender["default_rate"] = by_gender["default_rate"].map(lambda value: f"{value:.1%}")
print(by_gender.to_string())
print("Small groups have noisier rates, so the counts matter when comparing categories.")


## 6. Correlations with TARGET  (TODO(student))

Which numeric columns are most correlated with `TARGET` (positively and negatively)?
Remember the sign conventions: `DAYS_*` columns are negative numbers of days before the
application. Is a correlation of 0.1 large or small here?

In [ ]:
# Remove the employment sentinel before calculating numeric correlations
numeric = app.select_dtypes(include="number").drop(columns=["SK_ID_CURR"], errors="ignore").copy()
numeric.loc[numeric["DAYS_EMPLOYED"].eq(365243), "DAYS_EMPLOYED"] = float("nan")
correlations = numeric.corr(numeric_only=True)["TARGET"].drop("TARGET").dropna()
print("Most positive correlations with TARGET:")
print(correlations.sort_values(ascending=False).head(5).round(3).to_string())
print("\nMost negative correlations with TARGET:")
print(correlations.sort_values().head(5).round(3).to_string())
print("The correlations are modest; they show association, not causes, and "
      "0.1 is a small relationship.")


In [ ]:
# Summarize findings using the sample results above
miss = app.isna().mean().mul(100).sort_values(ascending=False)
anom = app["DAYS_EMPLOYED"].eq(365243)
gender_rates = (
    app.groupby("CODE_GENDER", dropna=False)["TARGET"]
    .mean()
    .sort_values(ascending=False)
)
corr = correlations.sort_values(
    key=lambda values: values.abs(), ascending=False
)
findings = [
    (
        f"The sample has {len(app):,} applicants and a "
        f"{app['TARGET'].mean():.1%} default rate, so defaults are the minority class."
    ),
    (
        f"{int((miss > 50).sum())} application columns are more than half empty; "
        "preprocessing needs to handle missing values."
    ),
    (
        f"{int(anom.sum()):,} applicants have the DAYS_EMPLOYED placeholder; "
        "I will replace it with missing and keep an indicator."
    ),
    (
        f"The CODE_GENDER group rates range from {gender_rates.min():.1%} "
        f"to {gender_rates.max():.1%}; counts should be considered and the "
        "column needs fairness review."
    ),
    (
        f"The largest absolute numeric correlation with TARGET is "
        f"{corr.index[0]} ({corr.iloc[0]:.3f}); this is an association, "
        "not a causal effect."
    ),
]
for finding in findings:
    print("- " + finding)
